# 01 · Phase 1 — Harm map (PopQA)
**Retrieval as Treatment**

Runs both arms — arm 0 = parametric answer, arm 1 = same prompt + top-k BM25 passages — on a popularity-stratified PopQA sample, then produces the harm map, the proxy-gate regret decomposition, and a paste-back report.

**Before you start**
- Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100 is ~5× faster).
- Everything generated is saved to Google Drive under `WORKDIR`, line by line. If the session dies, just re-run the notebook from the top: every step skips work that is already done.
- First-ever run downloads the model (~15 GB, or ~5.5 GB with the pre-quantized alternative in the config) and the BM25 index (~2.5 GB, persisted to Drive after the first download).

**What to send back:** the contents of `WORKDIR/results/phase1/report.md` (printed at the end) and the two `harm_map_*.png` figures.

In [ ]:
#@title 1 · Parameters + Drive + repo
REPO_URL = "https://github.com/<your-username>/retrieval-as-treatment"  #@param {type:"string"}
WORKDIR  = "/content/drive/MyDrive/retrieval-as-treatment"              #@param {type:"string"}
N_QUERIES = 2000  #@param {type:"integer"}
# Smoke test first? Set N_QUERIES = 100, run everything once, then set 2000 and re-run
# (the 100 already done are reused — nothing is recomputed).

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
os.makedirs(WORKDIR, exist_ok=True)
if not os.path.exists('/content/rat'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/rat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/rat', 'pull'], check=True)
print('repo ready at /content/rat')

In [ ]:
#@title 2 · System deps (Java 21 for Pyserini) + Python deps  — ~3-5 min the first time
!apt-get install -y -qq openjdk-21-jdk-headless > /dev/null 2>&1 || apt-get install -y -qq openjdk-21-jdk > /dev/null
!pip install -q -r /content/rat/requirements.txt
!pip install -q -e /content/rat
print('installed')

In [ ]:
#@title 3 · Environment
import os, glob, sys
jh = sorted(glob.glob('/usr/lib/jvm/java-21-openjdk*'))
assert jh, 'Java 21 not found — re-run the previous cell'
os.environ['JAVA_HOME'] = jh[-1]
os.environ['PATH'] = jh[-1] + '/bin:' + os.environ['PATH']
os.environ['HF_HOME'] = '/content/hf_cache'            # model weights: local disk (too big for free Drive)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
!java -version 2>&1 | head -1

sys.path.insert(0, '/content/rat')
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

from rat.config import load_config
from rat import phase1
cfg = load_config(WORKDIR, path='/content/rat/configs/phase1_popqa.yaml', n_queries=N_QUERIES)
cfg

In [ ]:
#@title 4 · Data — popularity-stratified PopQA sample (cached)
df, data_meta = phase1.step_data(cfg)
df.head()

In [ ]:
#@title 5 · Retrieval — BM25 top-10 per query (cached; index downloads once, then lives on Drive)
retr = phase1.step_retrieve(cfg, df)
q0 = df.qid.iloc[0]
print(df.question.iloc[0]); print(retr[q0][0]['title'], '|', retr[q0][0]['text'][:200])

In [ ]:
#@title 6 · Load the LLM (4-bit)
gen = phase1.make_generator(cfg)
gen.versions

In [ ]:
#@title 7 · Arm 0 — parametric answers + confidence probe (resumable)
gen0 = phase1.step_generate(cfg, df, arm=0, gen=gen)
print('arm0 mean F1 =', round(gen0.f1.mean(), 3), '| mean acc =', round(gen0.acc.mean(), 3))
gen0[['qid','answer','f1','acc','probe_maxprob_mean','probe_entropy_mean']].head()

In [ ]:
#@title 8 · Arm 1 — with top-k passages (resumable)
gen1 = phase1.step_generate(cfg, df, arm=1, gen=gen, retr_by_qid=retr)
print('arm1 mean F1 =', round(gen1.f1.mean(), 3), '| mean acc =', round(gen1.acc.mean(), 3))
gen1[['qid','answer','f1','acc','n_passages_used']].head()

In [ ]:
#@title 9 · Feature table (F0 + F1 tiers joined with both outcomes)
feat = phase1.step_features(cfg, df, gen0, gen1, retr)
feat[['qid','log_pop','probe_maxprob_mean','retr_top1','y0_f1','y1_f1','delta_f1','delta_acc']].describe().T

In [ ]:
#@title 10 · Analysis — harm map, marginals, proxy-gate regret; figures inline
out = phase1.step_analysis(cfg, feat, data_meta, gen.versions)
from IPython.display import Image, display
from rat.config import paths
import glob as _g
for p in sorted(_g.glob(os.path.join(paths(cfg)['figures'], '*.png'))):
    print(p); display(Image(p))

In [ ]:
#@title 11 · Report — copy everything below and send it back
print(open(os.path.join(paths(cfg)['results'], 'report.md')).read())

## Done
Files on Drive:
- `results/phase1/report.md`, `summary.json`, `gate_curve_*.csv`, `features_binned.parquet`, `versions.json`
- `figures/phase1/harm_map_f1.png`, `harm_map_acc.png`, `marginals.png`, `gate_regret_*.png`, `delta_hist.png`
- `logs/gen/*.jsonl` (every generation with probe stats), `retrievals/*.jsonl`, `features/*.parquet`

Second retriever tier (later): change `retriever` / `index_name` in the config and re-run from cell 5 — arm 0 is reused automatically, only arm 1 is regenerated.